In [6]:
from typing import List, Dict
from collections import defaultdict
from dataclasses import dataclass

@dataclass
class MCTSConfig:
    """Clase para mantener la configuración constante del problema"""
    students: List[Dict]
    challenges: List[Dict]
    careers: List[Dict]

    def __post_init__(self):
        # Preprocesar datos para acceso rápido
        self.career_limits = {career["Nombre"]: career["Maximo"] for career in self.careers}
        self.challenge_titles = {i+1: challenge["Titulo"] for i, challenge in enumerate(self.challenges)}
        self.n_challenges = len(self.challenges)

    def get_preference_score(self, student_idx: int, challenge_idx: int) -> float:
        """
        Calcula el score basado en las preferencias del estudiante.
        Este método no depende del estado, solo de la configuración.
        """
        challenge_title = self.challenge_titles[challenge_idx]
        student_preferences = self.students[student_idx]["Postulaciones"]

        try:
            preference_index = student_preferences.index(challenge_title)
            return 1 / (preference_index + 1)
        except ValueError:
            return 0

class Node:
    def __init__(self,
                 state: List[int],
                 config: MCTSConfig,
                 challenge_composition: Dict[int, Dict[str, int]] = None,
                 parent=None):
        """
        Inicializa un nodo del árbol MCTS.

        Args:
            state: Lista donde el índice es el estudiante y el valor es el desafío asignado
            config: Configuración compartida con todos los datos constantes
            challenge_composition: Composición actual de los desafíos
            parent: Nodo padre
        """
        self.state = state
        self.config = config
        self.parent = parent
        self.children = []

        # Atributos MCTS
        self.visits = 0
        self.value = 0.0

        # Inicializar composición de desafíos
        if challenge_composition is None:
            self.challenge_composition = {
                i+1: defaultdict(int) for i in range(config.n_challenges)
            }
            for student_idx, challenge in enumerate(state):
                if challenge != 0:
                    career = config.students[student_idx]["Carrera"]
                    self.challenge_composition[challenge][career] += 1
        else:
            self.challenge_composition = {
                challenge: defaultdict(int, composition)
                for challenge, composition in challenge_composition.items()
            }

    def evaluate_state(self) -> float:
        """
        Evalúa la calidad del estado actual.
        """
        if not self.is_terminal():
            return 0.0

        total_score = 0.0
        n_assignments = 0

        # Evaluar preferencias
        for student_idx, challenge in enumerate(self.state):
            if challenge != 0:
                preference_score = self.config.get_preference_score(student_idx, challenge)
                total_score += preference_score
                n_assignments += 1

        # Normalizar score de preferencias
        if n_assignments > 0:
            preference_score = total_score / n_assignments
        else:
            preference_score = 0

        # Evaluar tamaño de equipos
        team_size_score = 1.0
        team_sizes = defaultdict(int)
        for challenge in self.state:
            if challenge != 0:
                team_sizes[challenge] += 1

        for size in team_sizes.values():
            if size < 3:
                team_size_score *= 0.7
            elif size > 4:
                team_size_score *= 0.6

        return (0.7 * preference_score) + (0.3 * team_size_score)

    def get_possible_actions(self) -> List[int]:
        """Obtiene los desafíos posibles para el siguiente estudiante sin asignar."""
        if 0 not in self.state:
            return []

        current_student_idx = self.state.index(0)
        current_student_career = self.config.students[current_student_idx]["Carrera"]

        possible_challenges = []
        for challenge in range(1, self.config.n_challenges + 1):
            current_team_size = sum(self.challenge_composition[challenge].values())
            if current_team_size >= 4:
                continue

            career_count = self.challenge_composition[challenge][current_student_career]
            if career_count >= self.config.career_limits[current_student_career]:
                continue

            possible_challenges.append(challenge)

        return possible_challenges

    def apply_action(self, challenge: int) -> 'Node':
        """Aplica una acción y retorna el nuevo nodo hijo."""
        new_state = self.state.copy()
        current_student_idx = self.state.index(0)
        current_student_career = self.config.students[current_student_idx]["Carrera"]
        new_state[current_student_idx] = challenge

        new_composition = {
            c: defaultdict(int, comp)
            for c, comp in self.challenge_composition.items()
        }
        new_composition[challenge][current_student_career] += 1

        child = Node(
            state=new_state,
            config=self.config,  # Pasar la misma referencia
            challenge_composition=new_composition,
            parent=self
        )
        self.children.append(child)
        return child
    def is_terminal(self) -> bool:
        """
        Verifica si todos los estudiantes están asignados.
        Returns:
            bool: True si no hay más estudiantes por asignar (no hay 0s en el estado)
        """
        return 0 not in self.state
    def __str__(self) -> str:
        """
        Representación en string del nodo para debugging.
        """
        return f"Node(state={self.state}, visits={self.visits}, value={self.value:.2f})"

In [9]:
import random
from typing import List, Dict, Optional
import math
from dataclasses import dataclass
from collections import defaultdict

def mcts_assignment(students: List[Dict], challanges: List[Dict], careers: List[Dict], assignments: Optional[List[Dict]] = None) -> List[Dict]:
    """
    Asigna los estudiantes a desafíos mediante Monte Carlo Tree Search

    Args:
        students: Lista de estudiantes con Nombre,Carrera y Postulaciones
        challanges: Lista de desafíos con Título
        careers: Lista de carreras con Nombre y Máximo
        assignments: Lista de desafíos previamente asignados a los estudiantes

    Returns:
        list: Lista de diccionarios con el nombre del estudiante y el desafío asignado
    """
    # Inicializar configuración
    config = MCTSConfig(students=students, challenges=challanges, careers=careers)

    # Crear estado inicial
    if assignments is None:
        initial_state = [0] * len(students)
    else:
        initial_state = [0] * len(students)
        for i, assignment in enumerate(assignments):
            if assignment is not None:
                try:
                    challenge_idx = next(
                        idx + 1 for idx, ch in enumerate(challanges)
                        if ch["Titulo"] == assignment["Titulo"]
                    )
                    initial_state[i] = challenge_idx
                except StopIteration:
                    continue

    # Crear nodo raíz
    root = Node(state=initial_state, config=config)

    # Ejecutar MCTS hasta que todos los estudiantes estén asignados
    while not root.is_terminal():
        for _ in range(1000):  # Número de iteraciones por decisión
            node = root

            # Selection
            while node.get_possible_actions() == [] and node.children:
                uct_values = [
                    (child.value / child.visits if child.visits > 0 else float('inf')) +
                    1.414 * math.sqrt(math.log(node.visits) / child.visits if child.visits > 0 else float('inf'))
                    for child in node.children
                ]
                node = node.children[uct_values.index(max(uct_values))]

            # Expansion
            if node.get_possible_actions():
                action = random.choice(node.get_possible_actions())
                node = node.apply_action(action)

            # Simulation
            current_state = node.state.copy()
            current_composition = {
                c: defaultdict(int, comp)
                for c, comp in node.challenge_composition.items()
            }

            # Realizar simulación aleatoria hasta terminar
            while 0 in current_state:
                student_idx = current_state.index(0)
                student_career = config.students[student_idx]["Carrera"]

                valid_actions = []
                for challenge in range(1, config.n_challenges + 1):
                    team_size = sum(current_composition[challenge].values())
                    if team_size >= 4:
                        continue

                    career_count = current_composition[challenge][student_career]
                    if career_count >= config.career_limits[student_career]:
                        continue

                    valid_actions.append(challenge)

                if not valid_actions:
                    result = 0.0
                    break

                action = random.choice(valid_actions)
                current_state[student_idx] = action
                current_composition[action][student_career] += 1

            # Evaluar resultado de la simulación
            temp_node = Node(
                state=current_state,
                config=config,
                challenge_composition=current_composition
            )
            result = temp_node.evaluate_state()

            # Backpropagation
            while node is not None:
                node.visits += 1
                node.value += result
                node = node.parent

        # Seleccionar mejor acción para el estudiante actual
        if root.children:
            root = max(root.children, key=lambda c: c.visits)
        else:
            # No hay acciones válidas disponibles
            break

    # Convertir estado final al formato deseado
    final_assignments = []
    for student_idx, challenge_idx in enumerate(root.state):
        student_name = students[student_idx]["Nombre"]
        if challenge_idx == 0:
            final_assignments.append({
                "Nombre": student_name,
                "Desafio": None
            })
        else:
            final_assignments.append({
                "Nombre": student_name,
                "Desafio": config.challenge_titles[challenge_idx]
            })

    return final_assignments

In [10]:
import json
with open('body.json', 'r') as file:
    body = json.load(file)

In [11]:
from collections import defaultdict
from typing import List, Dict

def print_assignments_results(assignments: List[Dict], students: List[Dict], challanges: List[Dict]):
    """
    Imprime los resultados de las asignaciones de manera organizada, incluyendo desafíos no seleccionados

    Args:
        assignments: Lista de asignaciones {Nombre, Desafio}
        students: Lista original de estudiantes
        challanges: Lista original de desafíos
    """
    # Crear un diccionario de estudiantes para acceso rápido
    students_dict = {student["Nombre"]: student for student in students}

    # Crear conjunto de todos los desafíos disponibles
    all_challanges = {challange["Titulo"] for challange in challanges}

    # Agrupar asignaciones por desafío
    assignments_by_challange = defaultdict(list)
    for assignment in assignments:
        assignments_by_challange[assignment["Desafio"]].append(assignment["Nombre"])

    # Encontrar desafíos no seleccionados
    selected_challanges = set(assignments_by_challange.keys())
    unselected_challanges = all_challanges - selected_challanges

    def get_preference_number(student: Dict, challange: str) -> int:
        """Obtiene el número de preferencia del desafío para el estudiante"""
        try:
            return student["Postulaciones"].index(challange) + 1
        except ValueError:
            return -1

    print("\n=== RESULTADOS DE ASIGNACIÓN ===")

    # Ordenar desafíos alfabéticamente
    sorted_challanges = sorted(assignments_by_challange.keys())

    for challange in sorted_challanges:
        students_in_challange = assignments_by_challange[challange]

        print(f"\nDesafío: {challange}")
        print(f"Total de estudiantes: {len(students_in_challange)}")
        print("Integrantes:")

        # Ordenar estudiantes alfabéticamente dentro de cada desafío
        for student_name in sorted(students_in_challange):
            student = students_dict[student_name]
            preference = get_preference_number(student, challange)
            pref_text = f"preferencia {preference}" if preference > 0 else "sin preferencia"
            print(f"* {student_name} ({student['Carrera']}), con {pref_text}")

    # Imprimir desafíos no seleccionados
    if unselected_challanges:
        print("\n--- DESAFÍOS NO SELECCIONADOS ---")
        for challange in sorted(unselected_challanges):
            print(f"* {challange}")

In [12]:
import numpy as np
from collections import defaultdict, Counter

def calculate_metrics(students: list, challanges: list, assignments: list, careers: list) -> dict:
    """
    Calcula métricas de calidad para una asignación de estudiantes a desafíos

    Args:
        students: Lista de estudiantes con Nombre,Carrera y Postulaciones
        challanges: Lista de desafíos con Título y Carrerras
        assignments: Lista de desafíos asignados a los estudiantes
        careers: Lista de carreras con Nombre y Máximo

    Returns:
        dict: Diccionario con todas las métricas calculadas
    """
    metrics = {}

    # Crear diccionario de equipos
    equipos = defaultdict(list)
    for assignment in assignments:
        equipos[assignment['Desafio']].append(assignment['Nombre'])

    # Diccionario para mapear estudiante a sus postulaciones y carreras
    estudiantes_postulaciones = {s['Nombre']: s['Postulaciones'] for s in students}
    estudiantes_carreras = {s['Nombre']: s['Carrera'] for s in students}

    # Diccionario de límites por carrera
    limites_carreras = {c['Nombre']: c['Maximo'] for c in careers}

    # Calcular satisfacción individual y métricas relacionadas
    satisfacciones = []
    primera_prioridad = 0
    fuera_preferencias = 0

    for assignment in assignments:
        nombre = assignment['Nombre']
        desafio = assignment['Desafio']
        postulaciones = estudiantes_postulaciones[nombre]

        try:
            indice = postulaciones.index(desafio)
            satisfaccion = 1 / (indice + 1)
            if indice == 0:
                primera_prioridad += 1
        except ValueError:
            satisfaccion = 0
            fuera_preferencias += 1

        satisfacciones.append(satisfaccion)

    # Calcular tamaños de equipos y carreras por equipo
    tamanos_equipos = []
    carreras_por_equipo = []
    equipos_por_tamano = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
    equipos_exceden_limite = 0

    for desafio, miembros in equipos.items():
        if len(miembros) > 0:  # Solo considerar equipos con al menos un miembro
            tamano = len(miembros)
            tamanos_equipos.append(tamano)

            # Contar tamaños de equipo
            if tamano >= 5:
                equipos_por_tamano[5] += 1
            else:
                equipos_por_tamano[tamano] += 1

            # Contar carreras en el equipo
            carreras_equipo = Counter(estudiantes_carreras[estudiante] for estudiante in miembros)
            carreras_por_equipo.append(len(carreras_equipo))

            # Verificar límites por carrera
            excede_limite = False
            for carrera, cantidad in carreras_equipo.items():
                if cantidad > limites_carreras.get(carrera, 0):
                    excede_limite = True
                    break
            if excede_limite:
                equipos_exceden_limite += 1

    # Calcular desafíos sin equipo
    desafios_totales = set(c['Titulo'] for c in challanges)
    desafios_asignados = set(equipos.keys())
    desafios_sin_equipo = len(desafios_totales - desafios_asignados)

    # Almacenar métricas
    metrics['satisfaccion_promedio'] = np.mean(satisfacciones)
    metrics['estudiantes_primera_prioridad'] = primera_prioridad
    metrics['estudiantes_fuera_preferencias'] = fuera_preferencias
    metrics['desafios_sin_equipo'] = desafios_sin_equipo
    metrics['std_tamano_equipos'] = np.std(tamanos_equipos)
    metrics['promedio_carreras_por_equipo'] = np.mean(carreras_por_equipo)
    metrics['total_equipos'] = len([eq for eq in equipos.values() if len(eq) > 0])
    metrics['tamano_promedio_equipo'] = np.mean(tamanos_equipos)
    metrics['equipos_tamano_1'] = equipos_por_tamano[1]
    metrics['equipos_tamano_2'] = equipos_por_tamano[2]
    metrics['equipos_tamano_3'] = equipos_por_tamano[3]
    metrics['equipos_tamano_4'] = equipos_por_tamano[4]
    metrics['equipos_tamano_5_o_mas'] = equipos_por_tamano[5]
    metrics['equipos_exceden_limite_carrera'] = equipos_exceden_limite

    # Imprimir resultados
    print("\n=== Métricas de Asignación ===")
    print(f"Satisfacción promedio: {metrics['satisfaccion_promedio']:.3f}")
    print(f"Estudiantes en primera prioridad: {metrics['estudiantes_primera_prioridad']}")
    print(f"Estudiantes fuera de preferencias: {metrics['estudiantes_fuera_preferencias']}")
    print(f"Desafíos sin equipo: {metrics['desafios_sin_equipo']}")
    print(f"Desviación estándar tamaño equipos: {metrics['std_tamano_equipos']:.3f}")
    print(f"Promedio de carreras por equipo: {metrics['promedio_carreras_por_equipo']:.2f}")

    print(f"\nDistribución de tamaños de equipo:")
    print(f"Equipos de 1 estudiante: {metrics['equipos_tamano_1']}")
    print(f"Equipos de 2 estudiantes: {metrics['equipos_tamano_2']}")
    print(f"Equipos de 3 estudiantes: {metrics['equipos_tamano_3']}")
    print(f"Equipos de 4 estudiantes: {metrics['equipos_tamano_4']}")
    print(f"Equipos de 5 o más estudiantes: {metrics['equipos_tamano_5_o_mas']}")

    print(f"\nMétricas de límites:")
    print(f"Equipos que exceden límites por carrera: {metrics['equipos_exceden_limite_carrera']}")

    print(f"\nMétricas adicionales:")
    print(f"Total de equipos formados: {metrics['total_equipos']}")
    print(f"Tamaño promedio de equipo: {metrics['tamano_promedio_equipo']:.2f}")

    return metrics

In [45]:
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
import math
import random
from collections import defaultdict

@dataclass
class MCTSConfig:
    """Clase para mantener la configuración constante del problema"""
    students: List[Dict]
    challenges: List[Dict]
    careers: List[Dict]
    min_team_size: int = 3
    max_team_size: int = 4
    preference_weight: float = 0.7
    team_size_weight: float = 0.3
    exploration_constant: float = 1.414
    iterations_per_decision: int = 1000
    preference_decay: float = 1.0  # Factor de decaimiento para preferencias
    top_choice_bonus: float = 0.0  # Bonus para primera preferencia

    def __post_init__(self):
        self.career_limits = {career["Nombre"]: career["Maximo"] for career in self.careers}
        self.challenge_titles = {i+1: challenge["Titulo"] for i, challenge in enumerate(self.challenges)}
        self.n_challenges = len(self.challenges)
        
        #  Precalcular todas las preferencias
        self.preference_scores = {}
        for student_idx, student in enumerate(self.students):
            for challenge_idx in range(1, self.n_challenges + 1):
                self.preference_scores[(student_idx, challenge_idx)] = self._calculate_preference_score(
                    student["Postulaciones"], 
                    self.challenge_titles[challenge_idx]
                )

    def _calculate_preference_score(self, preferences: List[str], challenge_title: str) -> float:
        """
        Calcula el score de preferencia con opciones de personalización
        
        Args:
            preferences: Lista de títulos de desafíos en orden de preferencia
            challenge_title: Título del desafío a evaluar
            
        Returns:
            float: Score de preferencia entre 0 y 1 + bonus
        """
        try:
            preference_index = preferences.index(challenge_title)
            
            # Score base con decaimiento exponencial personalizable
            base_score = 1 / (1 + preference_index * self.preference_decay)
            
            # Bonus para primera preferencia si está configurado
            if preference_index == 0 and self.top_choice_bonus > 0:
                base_score += self.top_choice_bonus
                
            return base_score
            
        except ValueError:
            return 0

    def get_preference_score(self, student_idx: int, challenge_idx: int) -> float:
        """Obtiene el score precalculado de preferencia"""
        return self.preference_scores.get((student_idx, challenge_idx), 0)

class Node:
    def __init__(self,
                 state: List[int],
                 config: MCTSConfig,
                 challenge_composition: Optional[Dict[int, Dict[str, int]]] = None,
                 parent: Optional['Node'] = None):
        self.state = state
        self.config = config
        self.parent = parent
        self.children = []
        self.visits = 0
        self.value = 0.0
        self.untried_actions: Optional[List[int]] = None  # Cache for possible actions
        
        # Inicializar o copiar composición de desafíos
        if challenge_composition is None:
            self.challenge_composition = {
                i+1: defaultdict(int) for i in range(config.n_challenges)
            }
            for student_idx, challenge in enumerate(state):
                if challenge != 0:
                    career = config.students[student_idx]["Carrera"]
                    self.challenge_composition[challenge][career] += 1
        else:
            self.challenge_composition = {
                challenge: defaultdict(int, composition)
                for challenge, composition in challenge_composition.items()
            }

    def evaluate_state(self) -> float:
        """
        Evalúa el estado con penalizaciones severas para equipos inválidos
        y mayor peso para preferencias
        """
        if not self.is_terminal():
            return 0.0

        # Verificar tamaños de equipo
        team_sizes = defaultdict(int)
        for challenge in self.state:
            if challenge != 0:
                team_sizes[challenge] += 1

        # Penalización severa por equipos inválidos
        for size in team_sizes.values():
            if size < 3 or size > 4:
                return 0.0

        # Calcular score de preferencias
        preference_scores = []
        for student_idx, challenge in enumerate(self.state):
            if challenge != 0:
                score = self.config.get_preference_score(student_idx, challenge)
                # Bonus extra para primera preferencia
                if score > 0.9:  # Es primera preferencia
                    score *= 1.5
                preference_scores.append(score)

        if not preference_scores:
            return 0.0

        avg_preference_score = sum(preference_scores) / len(preference_scores)
        return avg_preference_score
    
    def _is_valid_assignment(self, challenge: int, student_career: str) -> bool:
        """Verifica si una asignación es válida según restricciones de carrera y tamaño"""
        team_size = sum(self.challenge_composition[challenge].values())
        if team_size >= self.config.max_team_size:
            return False
            
        career_count = self.challenge_composition[challenge][student_career]
        if career_count >= self.config.career_limits[student_career]:
            return False
            
        return True
    
    def get_possible_actions(self) -> List[int]:
        """
        Obtiene los desafíos posibles con restricciones estrictas de tamaño
        """
        if self.is_terminal():
            return []

        current_student_idx = self.state.index(0)
        current_student_career = self.config.students[current_student_idx]["Carrera"]
        
        # Calcular estadísticas actuales
        team_sizes = defaultdict(int)
        for challenge in self.state:
            if challenge != 0:
                team_sizes[challenge] += 1

        total_students = len(self.state)
        remaining_students = self.state.count(0)
        target_teams = math.ceil(total_students / 3)  # Intentar formar equipos de 3
        current_teams = len([size for size in team_sizes.values() if size > 0])

        possible_challenges = []
        student_preferences = self.config.students[current_student_idx]["Postulaciones"]

        # 1. Primera prioridad: Completar equipos existentes que tienen 2 estudiantes
        teams_of_two = [
            challenge for challenge, size in team_sizes.items() 
            if size == 2
        ]
        if teams_of_two:
            for challenge in teams_of_two:
                if self._is_valid_assignment(challenge, current_student_career):
                    possible_challenges.append(challenge)
            if possible_challenges:
                return possible_challenges

        # 2. Segunda prioridad: Unirse a equipos de 3 si quedan pocos estudiantes
        if remaining_students <= current_teams:
            teams_of_three = [
                challenge for challenge, size in team_sizes.items() 
                if size == 3
            ]
            for challenge in teams_of_three:
                if self._is_valid_assignment(challenge, current_student_career):
                    possible_challenges.append(challenge)
            if possible_challenges:
                return possible_challenges

        # 3. Tercera prioridad: Unirse a equipos existentes de tamaño válido
        existing_valid_teams = [
            challenge for challenge, size in team_sizes.items() 
            if size in [2, 3]
        ]
        if existing_valid_teams:
            preferred_teams = []
            other_teams = []
            for challenge in existing_valid_teams:
                if not self._is_valid_assignment(challenge, current_student_career):
                    continue
                if self.config.challenge_titles[challenge] in student_preferences:
                    preferred_teams.append(challenge)
                else:
                    other_teams.append(challenge)
            if preferred_teams:
                return preferred_teams
            if other_teams:
                return other_teams

        # 4. Cuarta prioridad: Crear nuevo equipo si es necesario
        if current_teams < target_teams:
            for challenge in range(1, self.config.n_challenges + 1):
                if team_sizes[challenge] == 0 and self._is_valid_assignment(challenge, current_student_career):
                    if self.config.challenge_titles[challenge] in student_preferences:
                        possible_challenges.append(challenge)
            if possible_challenges:
                return possible_challenges
            
            # Si no hay preferidos disponibles, considerar otros desafíos
            for challenge in range(1, self.config.n_challenges + 1):
                if team_sizes[challenge] == 0 and self._is_valid_assignment(challenge, current_student_career):
                    possible_challenges.append(challenge)

        return possible_challenges
    
    def select_child(self) -> Tuple[Optional['Node'], float]:
        """Selecciona el mejor hijo usando UCT mejorado"""
        if not self.children:
            return None, float('-inf')

        max_uct = float('-inf')
        best_child = None

        for child in self.children:
            if child.visits == 0:
                return child, float('inf')

            # UCT con componente de diversidad
            exploitation = child.value / child.visits
            exploration = self.config.exploration_constant * math.sqrt(math.log(self.visits) / child.visits)
            
            # Añadir componente de diversidad basado en la distribución de equipos
            diversity_score = self._calculate_diversity_score(child)
            
            uct = exploitation + exploration + (0.1 * diversity_score)

            if uct > max_uct:
                max_uct = uct
                best_child = child

        return best_child, max_uct

    def _calculate_diversity_score(self, node: 'Node') -> float:
        """Calcula un score de diversidad basado en la distribución de carreras"""
        diversity_scores = []
        
        for challenge in range(1, self.config.n_challenges + 1):
            team_size = sum(node.challenge_composition[challenge].values())
            if team_size == 0:
                continue
                
            career_counts = node.challenge_composition[challenge]
            unique_careers = len([c for c, count in career_counts.items() if count > 0])
            diversity_scores.append(unique_careers / team_size)
            
        return sum(diversity_scores) / len(diversity_scores) if diversity_scores else 0
    
    def apply_action(self, challenge: int) -> 'Node':
        """Aplica una acción y retorna el nuevo nodo hijo."""
        new_state = self.state.copy()
        current_student_idx = self.state.index(0)
        current_student_career = self.config.students[current_student_idx]["Carrera"]
        new_state[current_student_idx] = challenge

        new_composition = {
            c: defaultdict(int, comp)
            for c, comp in self.challenge_composition.items()
        }
        new_composition[challenge][current_student_career] += 1

        child = Node(
            state=new_state,
            config=self.config,  # Pasar la misma referencia
            challenge_composition=new_composition,
            parent=self
        )
        self.children.append(child)
        return child
    
    def is_terminal(self) -> bool:
        """
        Verifica si todos los estudiantes están asignados.
        Returns:
            bool: True si no hay más estudiantes por asignar (no hay 0s en el estado)
        """
        return 0 not in self.state
    
    def __str__(self) -> str:
        """
        Representación en string del nodo para debugging.
        """
        return f"Node(state={self.state}, visits={self.visits}, value={self.value:.2f})"

def mcts_assignment(
    students: List[Dict],
    challenges: List[Dict],
    careers: List[Dict],
    assignments: Optional[List[Dict]] = None,
    config_params: Optional[Dict] = None
) -> List[Dict]:
    """
    MCTS con parámetros optimizados para balance entre tamaño y preferencias
    """
    base_params = {
        'min_team_size': 3,
        'max_team_size': 4,
        'preference_weight': 0.8,  # Mayor peso a preferencias
        'team_size_weight': 0.2,
        'iterations_per_decision': 3000,
        'exploration_constant': 1.2  # Ajustado para mejor exploración
    }
    
    if config_params:
        base_params.update(config_params)
    
    result = mcts_core(students, challenges, careers, assignments, base_params)
    
    # Verificar si hay equipos inválidos
    team_sizes = defaultdict(int)
    for assignment in result:
        if assignment["Desafio"]:
            team_sizes[assignment["Desafio"]] += 1
            
    # Si hay equipos inválidos, intentar de nuevo con parámetros más estrictos
    if any(size < 3 or size > 4 for size in team_sizes.values()):
        base_params['team_size_weight'] = 0.9
        base_params['preference_weight'] = 0.1
        base_params['iterations_per_decision'] = 4000
        result = mcts_core(students, challenges, careers, assignments, base_params)
    
    return result

def mcts_core(
    students: List[Dict],
    challenges: List[Dict],
    careers: List[Dict],
    assignments: Optional[List[Dict]] = None,
    config_params: Optional[Dict] = None
) -> List[Dict]:
    """
    Implementación del núcleo del algoritmo Monte Carlo Tree Search (MCTS)
    
    Args:
        students: Lista de estudiantes con sus preferencias
        challenges: Lista de desafíos disponibles
        careers: Lista de carreras y sus límites
        assignments: Asignaciones previas (opcional)
        config_params: Parámetros de configuración
        
    Returns:
        List[Dict]: Lista de asignaciones finales
    """
    # 1. Inicialización
    config = MCTSConfig(
        students=students,
        challenges=challenges,
        careers=careers,
        **config_params
    )
    
    # Crear estado inicial
    initial_state = [0] * len(students)  # 0 significa no asignado
    if assignments:
        for i, assignment in enumerate(assignments):
            if assignment is not None:
                try:
                    challenge_idx = next(
                        idx + 1 for idx, ch in enumerate(challenges)
                        if ch["Titulo"] == assignment["Titulo"]
                    )
                    initial_state[i] = challenge_idx
                except StopIteration:
                    continue

    # Crear nodo raíz
    root = Node(state=initial_state, config=config)
    
    # 2. Ciclo principal de MCTS
    while not root.is_terminal():
        # Realizar múltiples iteraciones para cada decisión
        for _ in range(config.iterations_per_decision):
            node = root  # Comenzar desde la raíz
            
            # FASE 1: SELECTION (Selección)
            # Bajar por el árbol usando UCT hasta llegar a un nodo no completamente expandido
            while node.get_possible_actions() == [] and not node.is_terminal():
                child, _ = node.select_child()  # Seleccionar mejor hijo según UCT
                if child is None:
                    break
                node = child
            
            # FASE 2: EXPANSION (Expansión)
            # Si el nodo no es terminal, expandirlo
            if not node.is_terminal():
                actions = node.get_possible_actions()
                if actions:
                    action = random.choice(actions)  # Elegir acción aleatoria
                    node = node.apply_action(action)  # Crear nuevo nodo hijo
            
            # FASE 3: SIMULATION (Simulación)
            # Realizar una simulación desde el nodo actual hasta un estado terminal
            final_state = simulate_random_playout(node, config)
            
            # FASE 4: BACKPROPAGATION (Retropropagación)
            # Propagar el resultado hacia arriba en el árbol
            if final_state:
                result = Node(
                    state=final_state,
                    config=config,
                    challenge_composition=node.challenge_composition
                ).evaluate_state()
            else:
                result = 0.0
                
            # Actualizar estadísticas en cada nodo del camino
            while node is not None:
                node.visits += 1
                node.value += result
                node = node.parent
        
        # 3. Selección de mejor acción
        # Elegir la mejor acción basada en las visitas (no en UCT)
        best_child = None
        best_visits = -1
        
        for child in root.children:
            if child.visits > best_visits:
                best_visits = child.visits
                best_child = child
        
        if best_child:
            root = best_child  # Avanzar al mejor hijo
        else:
            break  # No hay más acciones posibles

    # 4. Convertir estado final a formato de salida
    final_assignments = []
    for student_idx, challenge_idx in enumerate(root.state):
        student_name = students[student_idx]["Nombre"]
        if challenge_idx == 0:
            final_assignments.append({
                "Nombre": student_name,
                "Desafio": None
            })
        else:
            final_assignments.append({
                "Nombre": student_name,
                "Desafio": config.challenge_titles[challenge_idx]
            })

    return final_assignments

def simulate_random_playout(node: Node, config: MCTSConfig) -> Optional[List[int]]:
    """
    Simula un playout con énfasis en formación de equipos válidos
    """
    current_state = node.state.copy()
    current_composition = {
        c: defaultdict(int, comp)
        for c, comp in node.challenge_composition.items()
    }

    def get_team_priority(challenge: int, team_sizes: Dict[int, int]) -> float:
        size = team_sizes[challenge]
        if size == 0:
            return 0.5  # Baja prioridad para equipos nuevos
        elif size < 3:
            return 3.0  # Alta prioridad para completar equipos
        elif size == 3:
            return 1.0  # Prioridad normal para equipos de 3
        else:
            return 0.0  # No permitir equipos más grandes

    while 0 in current_state:
        student_idx = current_state.index(0)
        student_career = config.students[student_idx]["Carrera"]
        
        # Calcular tamaños actuales
        team_sizes = defaultdict(int)
        for assignment in current_state:
            if assignment != 0:
                team_sizes[assignment] += 1

        # Obtener acciones válidas y sus pesos
        valid_actions = []
        weights = []
        
        for challenge in range(1, config.n_challenges + 1):
            if team_sizes[challenge] >= 4:
                continue
                
            career_count = current_composition[challenge][student_career]
            if career_count >= config.career_limits[student_career]:
                continue

            valid_actions.append(challenge)
            
            # Calcular peso combinando prioridad de equipo y preferencia
            team_priority = get_team_priority(challenge, team_sizes)
            preference_score = config.get_preference_score(student_idx, challenge)
            
            weight = team_priority * (1 + preference_score)
            weights.append(weight)

        if not valid_actions:
            return None

        action = random.choices(valid_actions, weights=weights, k=1)[0]
        current_state[student_idx] = action
        current_composition[action][student_career] += 1

    return current_state

In [46]:
config_params={
    'min_team_size': 3,
    'max_team_size': 4,
    'preference_weight': 0.8,  # Dar más peso a preferencias
    'team_size_weight': 0.2,
    'iterations_per_decision': 2000  # Más iteraciones para mejor exploración
}

'''
config_params: Parámetros de configuración que pueden incluir:
            - preference_weight: Peso para las preferencias (default: 0.7)
            - team_size_weight: Peso para tamaño de equipos (default: 0.3)
            - preference_decay: Factor de decaimiento para preferencias (default: 1.0)
            - top_choice_bonus: Bonus para primera preferencia (default: 0.0)
'''
assignment = mcts_assignment(body["estudiantes"], body["desafios"], body["carreras"], config_params=config_params)
calculate_metrics(students=body["estudiantes"], challanges=body["desafios"],assignments=assignment, careers=body["carreras"])


=== Métricas de Asignación ===
Satisfacción promedio: 0.174
Estudiantes en primera prioridad: 7
Estudiantes fuera de preferencias: 47
Desafíos sin equipo: 16
Desviación estándar tamaño equipos: 8.361
Promedio de carreras por equipo: 1.09

Distribución de tamaños de equipo:
Equipos de 1 estudiante: 22
Equipos de 2 estudiantes: 0
Equipos de 3 estudiantes: 0
Equipos de 4 estudiantes: 0
Equipos de 5 o más estudiantes: 1

Métricas de límites:
Equipos que exceden límites por carrera: 1

Métricas adicionales:
Total de equipos formados: 23
Tamaño promedio de equipo: 2.78


{'satisfaccion_promedio': 0.17447916666666666,
 'estudiantes_primera_prioridad': 7,
 'estudiantes_fuera_preferencias': 47,
 'desafios_sin_equipo': 16,
 'std_tamano_equipos': 8.361175919685245,
 'promedio_carreras_por_equipo': 1.0869565217391304,
 'total_equipos': 23,
 'tamano_promedio_equipo': 2.782608695652174,
 'equipos_tamano_1': 22,
 'equipos_tamano_2': 0,
 'equipos_tamano_3': 0,
 'equipos_tamano_4': 0,
 'equipos_tamano_5_o_mas': 1,
 'equipos_exceden_limite_carrera': 1}

In [39]:
assignment

[{'Nombre': 'Matias Ignacio Ayala Nanjari',
  'Desafio': 'SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.'},
 {'Nombre': 'Ignacio Andrés Araya Salinas',
  'Desafio': 'SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.'},
 {'Nombre': 'Camila Fernanda Guerrero Bustamante',
  'Desafio': 'SimCity: Creando una simulación 3D de la ciudad para el estudio de fenómenos naturales.'},
 {'Nombre': 'Simón Josías Cristóbal Álvarez Díaz',
  'Desafio': 'Modelo Acústico de Lenguaje Natural (NLP) Para Conversaciones Oncológicas de Salud'},
 {'Nombre': 'Fernanda Ivonne Viera Castillo    ',
  'Desafio': 'Modelo Acústico de Lenguaje Natural (NLP) Para Conversaciones Oncológicas de Salud'},
 {'Nombre': 'Manuel Cruces Pirce',
  'Desafio': 'Modelo Acústico de Lenguaje Natural (NLP) Para Conversaciones Oncológicas de Salud'},
 {'Nombre': 'Carlos Alfredo Cea Rios',
  'Desafio': 'Sistema para el control y mejora de la limpieza industria

In [31]:
print_assignments_results(assignment, body["estudiantes"], body["desafios"])


=== RESULTADOS DE ASIGNACIÓN ===


TypeError: '<' not supported between instances of 'NoneType' and 'str'